# ============================================================
# EXTENDED ISOLATION FOREST — PREPROCESADO FFT
# Features: espectro dB normalizado (0–1kHz) por señal
# 8 señales × 1001 bins = 8008 features por ventana
# Umbral: media + 3*std (paper)
# Agregación por experimento (mediana de 100 ventanas)
# ============================================================

In [ ]:
# ============================================================
# 0. INSTALACIÓN
# ============================================================

!pip install h2o optuna optuna-dashboard plotly seaborn

In [ ]:
# ============================================================
# 1. IMPORTS
# ============================================================

import h2o
from h2o.estimators import H2OExtendedIsolationForestEstimator
import optuna
from optuna.trial import TrialState
import optuna.visualization as vis
import numpy as np
import pandas as pd
import os
import json
import seaborn as sns
import matplotlib.pyplot as plt
import plotly.io as pio
pio.renderers.default = "browser"

In [ ]:
# ============================================================
# 2. CONFIGURACIÓN
# ============================================================

RUTA_FEATURES   = "features_fft"          # <-- carpeta nueva con preprocesado FFT
CSV_INDEX       = "Motor_DB/index/master_index.csv"
RUTA_RESULTADOS = "resultados_fft"
os.makedirs(RUTA_RESULTADOS, exist_ok=True)

N_TRIALS          = 500
SEED              = 42
VENTANAS_POR_EXP  = 100   # 100 segundos a 20kHz, ventanas de 1s

# El extension_level máximo de EIF = n_features - 1
# Se calcula dinámicamente tras leer las columnas (ver celda siguiente)

VELOCIDADES_ESTABLES = {"S1500", "S1200", "S900"}  # regímenes estables (excluye STrap, SSteps, SStart y S20)

# Optuna buscará el percentil óptimo en este rango
PERCENTIL_MIN = 90
PERCENTIL_MAX = 99
CONTROLES_VALIDOS = {"d", "s"}  # excluye grid directo (l): sub-representado en train


In [ ]:
# ============================================================
# 3. LEER COLUMNAS Y CALCULAR EXT_MAX
# ============================================================

cols_todas      = pd.read_csv(os.path.join(RUTA_FEATURES, "train", "sano_train.csv"), nrows=0).columns.tolist()
cols_electricas = [c for c in cols_todas if c.startswith(("u_bin", "v_bin", "w_bin"))]
cols_vibracion  = [c for c in cols_todas if c not in cols_electricas]

# Eléctricas: 3 señales × 201 bins = 603
# Vibración : 5 señales × 1001 bins = 5005
# Total     : 5608 features
N_FEATURES   = len(cols_todas)
EXT_MAX      = N_FEATURES - 1

print(f"Total features   : {N_FEATURES}  (eléctricas hasta 200Hz, vibración hasta 1000Hz)")
print(f"  Eléctricas     : {len(cols_electricas)}  (u, v, w × 201 bins — 0 a 200 Hz)")
print(f"  Vibración      : {len(cols_vibracion)}  (5 señales × 1001 bins — 0 a 1000 Hz)")
print(f"EXT_MAX          : {EXT_MAX}")

EXT_MAX_BUSQUEDA = min(EXT_MAX, 100)
print(f"EXT_MAX búsqueda : {EXT_MAX_BUSQUEDA}")

In [ ]:
# ============================================================
# 4. INICIALIZAR H2O
# ============================================================

# Con 8008 features y 100 ventanas/experimento los frames son grandes.
# Ajusta max_mem_size según tu RAM disponible.
h2o.init(
    nthreads    = -1,
    max_mem_size = "12G"
)

In [ ]:
# ============================================================
# 5. CARGAR DATOS
# ============================================================

print("Cargando datos...\n")

index = pd.read_csv(CSV_INDEX)

# Filtrar transitorios — mismo criterio que el preprocesado FFT
index = index[index["Velocidad"].astype(str).isin(VELOCIDADES_ESTABLES)]
index = index[index["Control"].astype(str).isin(CONTROLES_VALIDOS)].reset_index(drop=True)
print(f"Index tras filtros: {len(index)} experimentos (sin transitorios ni grid directo)\n")

train     = h2o.import_file(os.path.join(RUTA_FEATURES, "train", "sano_train.csv"))
val       = h2o.import_file(os.path.join(RUTA_FEATURES, "val",   "sano.csv"))
test_sano = h2o.import_file(os.path.join(RUTA_FEATURES, "test",  "sano.csv"))

carpeta_test  = os.path.join(RUTA_FEATURES, "test")
fallos_frames = {}
for archivo in sorted(os.listdir(carpeta_test)):
    if archivo.endswith(".csv") and archivo != "sano.csv":
        nombre = archivo.replace(".csv", "")
        fallos_frames[nombre] = h2o.import_file(os.path.join(carpeta_test, archivo))

print(f"Train     : {train.shape[0]} ventanas  ({train.shape[0] // VENTANAS_POR_EXP} experimentos)")
print(f"Val sano  : {val.shape[0]} ventanas  ({val.shape[0] // VENTANAS_POR_EXP} experimentos)")
print(f"Test sano : {test_sano.shape[0]} ventanas  ({test_sano.shape[0] // VENTANAS_POR_EXP} experimentos)")
print(f"Grupos de fallo: {len(fallos_frames)}")

VELOCIDADES_ESTABLES = {"S1500", "S1200", "S900"}  # regímenes estables (excluye STrap, SSteps, SStart y S20)

# Optuna buscará el percentil óptimo en este rango
PERCENTIL_MIN = 90
PERCENTIL_MAX = 99


In [ ]:
# ============================================================
# 6. FUNCIÓN DE AGREGACIÓN POR EXPERIMENTO
# ============================================================

def agregar_por_experimento(scores_array, ventanas_por_exp):
    """
    Divide los scores en bloques de ventanas_por_exp ventanas
    y devuelve la mediana de cada bloque (= 1 score por experimento).
    """
    medianas = []
    for i in range(0, len(scores_array), ventanas_por_exp):
        grupo = scores_array[i : i + ventanas_por_exp]
        if len(grupo) > 0:
            medianas.append(np.median(grupo))
    return np.array(medianas)

def agregar_variable(scores_array, n_archivos):
    """
    Para grupos con distinto nº de ventanas: divide equitativamente
    el total entre n_archivos.
    """
    ventanas_por_exp = len(scores_array) // n_archivos
    return agregar_por_experimento(scores_array, ventanas_por_exp)

In [ ]:
# ============================================================
# 7. FUNCIÓN OBJETIVO OPTUNA
# ============================================================
# Dos hiperparámetros nuevos:
#   percentil_umbral : Optuna elige entre PERCENTIL_MIN y PERCENTIL_MAX
#
# Función objetivo: minimizar (std_sanos / rango_sanos)
#   → cuanto más compacta la distribución de sanos, mejor.
#   → el umbral queda como percentil sobre los sanos, no como media+k·std.

def objective(trial):

    ntrees           = trial.suggest_int("ntrees",           30,  150)
    sample_size      = trial.suggest_int("sample_size",      32,  128, step=32)
    extension_level  = trial.suggest_int("extension_level",  0,   50)
    percentil_umbral = trial.suggest_int("percentil_umbral", 90,  99)

    model = H2OExtendedIsolationForestEstimator(
        ntrees          = ntrees,
        sample_size     = sample_size,
        extension_level = extension_level,
        seed            = SEED
    )
    try:
        model.train(training_frame=train)
    except Exception as e:
        print(f"  Trial fallido: {e}")
        raise optuna.exceptions.TrialPruned()

    s_train = model.predict(train)["anomaly_score"].as_data_frame().values.flatten()
    s_val   = model.predict(val)["anomaly_score"].as_data_frame().values.flatten()

    med_train = agregar_por_experimento(s_train, VENTANAS_POR_EXP)
    med_val   = agregar_por_experimento(s_val,   VENTANAS_POR_EXP)
    todos     = np.concatenate([med_train, med_val])

    media  = np.mean(todos)
    std    = np.std(todos)
    rango  = np.max(todos) - np.min(todos)
    umbral = np.percentile(todos, percentil_umbral)

    # Compacidad relativa: std / rango → minimizar
    # Cuanto más pequeño, más juntos están los sanos → umbral más bajo → más sensible
    compacidad = std / (rango + 1e-12)

    trial.set_user_attr("media",            round(float(media),      6))
    trial.set_user_attr("std",              round(float(std),        6))
    trial.set_user_attr("umbral",           round(float(umbral),     6))
    trial.set_user_attr("percentil_umbral", percentil_umbral)
    trial.set_user_attr("compacidad",       round(float(compacidad), 6))

    return compacidad  # minimizar → sanos más compactos → mejor separación con fallos


In [ ]:
h2o.cluster().show_status()   # verificar que H2O sigue vivo antes de empezar

In [ ]:
# ============================================================
# 8. OPTIMIZACIÓN OPTUNA
# ============================================================

print("\n" + "="*60)
print(f"Iniciando optimización: {N_TRIALS} trials")
print(f"Features por ventana : {N_FEATURES}")
print("="*60 + "\n")

sampler = optuna.samplers.TPESampler(seed=SEED)
study   = optuna.create_study(
    direction      = "minimize",
    sampler        = sampler,
    storage        = f"sqlite:///{RUTA_RESULTADOS}/optuna_eif_fft.db",
    study_name     = "EIF_motores_fft_8008features",
    load_if_exists = True
)
#study.optimize(objective, n_trials=N_TRIALS, show_progress_bar=True)
study.optimize(objective, n_trials=10, show_progress_bar=True)

In [ ]:
# ============================================================
# 9. RESULTADOS OPTUNA
# ============================================================

complete_trials = study.get_trials(deepcopy=False, states=[TrialState.COMPLETE])

print("\n" + "="*60)
print("RESULTADOS OPTUNA")
print("="*60)
print(f"  Trials completados : {len(complete_trials)}")

best = study.best_trial
print(f"\n  Mejor compacidad   : {best.value:.6f}")
print(f"  Umbral             : {best.user_attrs['umbral']:.6f}")
print(f"  Percentil umbral   : {best.user_attrs['percentil_umbral']}")
print(f"  Media sanos        : {best.user_attrs['media']:.6f}")
print(f"  Std sanos          : {best.user_attrs['std']:.6f}")
print(f"\n  Mejores hiperparámetros:")
for k, v in best.params.items():
    print(f"    {k}: {v}")

with open(os.path.join(RUTA_RESULTADOS, "mejores_hiperparametros.json"), "w") as f:
    json.dump(best.params, f, indent=2)

In [ ]:
# ============================================================
# 10. MODELO FINAL
# ============================================================

# Reiniciar H2O para liberar memoria antes del entrenamiento final
h2o.cluster().shutdown(prompt=False)
import time
time.sleep(5)

h2o.init(nthreads=-1, max_mem_size="16G")

# Recargar index con los mismos filtros que en la celda 5
index = pd.read_csv(CSV_INDEX)
index = index[index["Velocidad"].astype(str).isin(VELOCIDADES_ESTABLES)]
index = index[index["Control"].astype(str).isin(CONTROLES_VALIDOS)].reset_index(drop=True)
print(f"Index recargado: {len(index)} experimentos\n")

# Recargar datos
train     = h2o.import_file(os.path.join(RUTA_FEATURES, "train", "sano_train.csv"))
val       = h2o.import_file(os.path.join(RUTA_FEATURES, "val",   "sano.csv"))
test_sano = h2o.import_file(os.path.join(RUTA_FEATURES, "test",  "sano.csv"))

fallos_frames = {}
for archivo in sorted(os.listdir(carpeta_test)):
    if archivo.endswith(".csv") and archivo != "sano.csv":
        nombre = archivo.replace(".csv", "")
        fallos_frames[nombre] = h2o.import_file(os.path.join(carpeta_test, archivo))

print("\n" + "="*60)
print("ENTRENANDO MODELO FINAL")
print("="*60)

# Recargar best desde Optuna por si se reinició el kernel
study = optuna.load_study(
    study_name = "EIF_motores_fft_8008features",
    storage    = f"sqlite:///{RUTA_RESULTADOS}/optuna_eif_fft.db"
)
best = study.best_trial

modelo_final = H2OExtendedIsolationForestEstimator(
    ntrees          = best.params["ntrees"],
    sample_size     = best.params["sample_size"],
    extension_level = best.params["extension_level"],
    seed            = SEED
)
modelo_final.train(training_frame=train)

# Calcular umbral final con train + val
s_train_f = modelo_final.predict(train)["anomaly_score"].as_data_frame().values.flatten()
s_val_f   = modelo_final.predict(val)["anomaly_score"].as_data_frame().values.flatten()

med_train_f = agregar_por_experimento(s_train_f, VENTANAS_POR_EXP)
med_val_f   = agregar_por_experimento(s_val_f,   VENTANAS_POR_EXP)
med_sanos   = np.concatenate([med_train_f, med_val_f])

# Recuperar percentil óptimo elegido por Optuna
percentil_optimo = best.user_attrs["percentil_umbral"]

media_sanos = np.mean(med_sanos)
std_sanos   = np.std(med_sanos)
umbral      = np.percentile(med_sanos, percentil_optimo)

print(f"\n  Umbral final (percentil {percentil_optimo}) : {umbral:.6f}")
print(f"  Media sanos                        : {media_sanos:.6f}")
print(f"  Std sanos                          : {std_sanos:.6f}")
print(f"  Equivale a media + {(umbral-media_sanos)/std_sanos:.2f}·std")

# Guardar umbral
with open(os.path.join(RUTA_RESULTADOS, "umbral.json"), "w") as f:
    json.dump({"umbral": umbral, "media": media_sanos, "std": std_sanos, "percentil": percentil_optimo}, f, indent=2)

VELOCIDADES_ESTABLES = {"S1500", "S1200", "S900"}  # regímenes estables (excluye STrap, SSteps, SStart y S20)
CONTROLES_VALIDOS = {"d", "s"}  # excluye grid directo (l): sub-representado en train


In [ ]:
# ============================================================
# 11. EVALUACIÓN POR EXPERIMENTO
# ============================================================

print("\n" + "="*60)
print("EVALUACIÓN POR EXPERIMENTO")
print("="*60)

resultados    = []
datos_boxplot = []

def evaluar_grupo(nombre, scores, n_archivos, es_fallo=True):
    medianas   = agregar_variable(scores, n_archivos)
    detectados = np.sum(medianas > umbral)
    total      = len(medianas)
    deteccion  = detectados / total * 100

    for m in medianas:
        datos_boxplot.append({"grupo": nombre, "mediana": m, "es_fallo": es_fallo})

    return {
        "grupo":          nombre,
        "n_experimentos": total,
        "detectados":     int(detectados),
        "deteccion_%":    round(deteccion, 1),
        "mediana_media":  round(float(np.mean(medianas)), 6),
        "mediana_max":    round(float(np.max(medianas)),  6),
    }

# Sano test
s_sano_test = modelo_final.predict(test_sano)["anomaly_score"].as_data_frame().values.flatten()
n_sano_test = len(index[(index["Split"] == "test") & (index["Maquina"] == "h")])
res_sano    = evaluar_grupo("sano_test", s_sano_test, n_sano_test, es_fallo=False)
resultados.append(res_sano)
print(f"\n  sano_test → {res_sano['detectados']}/{res_sano['n_experimentos']} "
      f"detectados como anomalía ({res_sano['deteccion_%']}%)  ← idealmente 0%")

# Fallos
print()
for nombre, frame in sorted(fallos_frames.items()):
    s_f    = modelo_final.predict(frame)["anomaly_score"].as_data_frame().values.flatten()
    n_arch = len(index[(index["Split"] == "test") & (index["Fallo"] == nombre)])
    res    = evaluar_grupo(nombre, s_f, n_arch, es_fallo=True)
    resultados.append(res)
    print(f"  {nombre:<55} "
          f"{res['detectados']:>2}/{res['n_experimentos']:>2}  "
          f"({res['deteccion_%']:>5.1f}%)")

# Añadir train y val al boxplot
for nombre, scores in [("sano_train", s_train_f), ("sano_val", s_val_f)]:
    for m in agregar_por_experimento(scores, VENTANAS_POR_EXP):
        datos_boxplot.append({"grupo": nombre, "mediana": m, "es_fallo": False})

# Guardar resultados
df_resultados = pd.DataFrame(resultados)
df_resultados.to_csv(os.path.join(RUTA_RESULTADOS, "resultados_por_experimento.csv"), index=False)
print(f"\nGuardado: {RUTA_RESULTADOS}/resultados_por_experimento.csv")

# Optuna buscará el percentil óptimo en este rango
PERCENTIL_MIN = 90
PERCENTIL_MAX = 99


In [ ]:
# ============================================================
# 12. BOXPLOT ESTILO PAPER
# ============================================================

df_box = pd.DataFrame(datos_boxplot)

grupos_sanos  = ["sano_train", "sano_val", "sano_test"]
grupos_fallos = sorted([n for n in df_box["grupo"].unique() if n not in grupos_sanos])
orden         = grupos_sanos + grupos_fallos
colores       = {g: "steelblue" if g in grupos_sanos else "salmon" for g in orden}

plt.figure(figsize=(24, 7))
sns.boxplot(
    data    = df_box[df_box["grupo"].isin(orden)],
    x       = "grupo",
    y       = "mediana",
    order   = orden,
    palette = colores
)
plt.axhline(y=umbral, color="red", linestyle="--", linewidth=1.5,
            label=f"Umbral = {umbral:.4f}")
plt.xticks(rotation=45, ha="right", fontsize=7)
plt.title("EIF FFT — Anomaly Score por experimento (mediana) | Sanos vs Fallos")
plt.xlabel("Grupo")
plt.ylabel("Mediana del Anomaly Score")
plt.legend()
plt.tight_layout()
plt.savefig(os.path.join(RUTA_RESULTADOS, "boxplot_resultados_fft.png"), dpi=150)
plt.show()

In [ ]:
# ============================================================
# 13. GRÁFICAS OPTUNA
# ============================================================

print("\nGenerando gráficas Optuna...")

fig1 = vis.plot_optimization_history(study)
fig1.update_layout(title="Evolución del umbral por trial")
fig1.write_html(os.path.join(RUTA_RESULTADOS, "optuna_historia.html"))
fig1.show()

fig2 = vis.plot_param_importances(study)
fig2.update_layout(title="Importancia de hiperparámetros")
fig2.write_html(os.path.join(RUTA_RESULTADOS, "optuna_importancia.html"))
fig2.show()

fig3 = vis.plot_parallel_coordinate(study)
fig3.update_layout(title="Coordenadas paralelas")
fig3.write_html(os.path.join(RUTA_RESULTADOS, "optuna_coordenadas.html"))
fig3.show()

fig4 = vis.plot_contour(study, params=["ntrees", "extension_level"])
fig4.update_layout(title="Contour: ntrees vs extension_level")
fig4.write_html(os.path.join(RUTA_RESULTADOS, "optuna_contour.html"))
fig4.show()

print("\n✅ Todo completado.")
print(f"Umbral final : {umbral:.6f}")
h2o.cluster().show_status()